In [1]:
import torch
import torch.nn as nn
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader

# Generate a binary classification dataset
X, y = make_moons(
    n_samples=1000,
    noise=0.2,
    random_state=42
)

# Split into training and test data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Scale features using training data only
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert NumPy arrays into PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
y_test = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

print(X_train.shape)
print(y_train.shape)

torch.Size([800, 2])
torch.Size([800, 1])


In [2]:
class ANN(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),

            nn.Linear(16, 8),
            nn.ReLU(),

            nn.Linear(8, 1)
        )

    def forward(self, x):
        return self.network(x)


model = ANN()

print(model)

ANN(
  (network): Sequential(
    (0): Linear(in_features=2, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=8, bias=True)
    (3): ReLU()
    (4): Linear(in_features=8, out_features=1, bias=True)
  )
)


In [3]:
dataset = TensorDataset(X_train, y_train)

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True
)

loss_fn = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [4]:
epochs = 100

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for X_batch, y_batch in loader:

        # Forward pass
        logits = model(X_batch)

        # Calculate loss
        loss = loss_fn(logits, y_batch)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()

        # Update weights
        optimizer.step()

        total_loss += loss.item() * len(X_batch)

    if (epoch + 1) % 10 == 0:
        avg_loss = total_loss / len(dataset)
        print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")

Epoch 10, Loss: 0.3570
Epoch 20, Loss: 0.2781
Epoch 30, Loss: 0.2275
Epoch 40, Loss: 0.1794
Epoch 50, Loss: 0.1425
Epoch 60, Loss: 0.1199
Epoch 70, Loss: 0.1065
Epoch 80, Loss: 0.0989
Epoch 90, Loss: 0.0942
Epoch 100, Loss: 0.0910


In [5]:
model.eval()

with torch.no_grad():
    logits = model(X_test)

    probabilities = torch.sigmoid(logits)

    predictions = (probabilities >= 0.5).float()

    accuracy = (predictions == y_test).float().mean()

print(f"Test Accuracy: {accuracy.item() * 100:.2f}%")

Test Accuracy: 98.50%


In [6]:
import numpy as np

# Two new feature values
new_sample = np.array([[0.5, 0.2]])

# Apply the SAME scaler used for training
new_sample = scaler.transform(new_sample)

new_sample = torch.tensor(
    new_sample,
    dtype=torch.float32
)

model.eval()

with torch.no_grad():
    logit = model(new_sample)
    probability = torch.sigmoid(logit).item()

predicted_class = int(probability >= 0.5)

print("Probability of Class 1:", round(probability, 4))
print("Predicted class:", predicted_class)

Probability of Class 1: 0.5701
Predicted class: 1
